<a href="https://colab.research.google.com/github/gitBarrettJones/CIS115-Spring2026-Assignments/blob/main/Week15AI2/Week15AI2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gardening Inventory Tool

In [1]:
import sqlite3
from datetime import datetime, timedelta

DATABASE_NAME = 'gardening_inventory.db'

def create_database_and_table():
    """Creates the SQLite database and the plants table if they don't exist."""
    conn = sqlite3.connect(DATABASE_NAME)
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS plants (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT NOT NULL UNIQUE,
            type TEXT NOT NULL,
            watering_frequency_days INTEGER NOT NULL,
            last_watered_date TEXT NOT NULL,
            next_watering_date TEXT
        )
    ''')
    conn.commit()
    conn.close()
    print(f"Database '{DATABASE_NAME}' and 'plants' table ensured.")

# Initialize the database
create_database_and_table()

Database 'gardening_inventory.db' and 'plants' table ensured.


Now that the database is set up, let's create functions to add new plants and update their watering status.

In [2]:
def calculate_next_watering_date(last_watered_str, frequency_days):
    """Calculates the next watering date."""
    last_watered = datetime.strptime(last_watered_str, '%Y-%m-%d').date()
    next_watering = last_watered + timedelta(days=frequency_days)
    return next_watering.strftime('%Y-%m-%d')

def add_plant(name, plant_type, watering_frequency_days, last_watered_date):
    """Adds a new plant to the database."""
    conn = sqlite3.connect(DATABASE_NAME)
    cursor = conn.cursor()
    try:
        next_watering = calculate_next_watering_date(last_watered_date, watering_frequency_days)
        cursor.execute(
            "INSERT INTO plants (name, type, watering_frequency_days, last_watered_date, next_watering_date) VALUES (?, ?, ?, ?, ?)",
            (name, plant_type, watering_frequency_days, last_watered_date, next_watering)
        )
        conn.commit()
        print(f"Plant '{name}' added successfully. Next watering due: {next_watering}")
    except sqlite3.IntegrityError:
        print(f"Error: Plant with name '{name}' already exists.")
    except ValueError as e:
        print(f"Error with date format: {e}. Please use YYYY-MM-DD.")
    finally:
        conn.close()

def update_last_watered(plant_name, new_last_watered_date):
    """Updates the last watered date and recalculates the next watering date for a plant."""
    conn = sqlite3.connect(DATABASE_NAME)
    cursor = conn.cursor()
    try:
        cursor.execute("SELECT watering_frequency_days FROM plants WHERE name = ?", (plant_name,))
        result = cursor.fetchone()
        if result:
            frequency_days = result[0]
            next_watering = calculate_next_watering_date(new_last_watered_date, frequency_days)
            cursor.execute(
                "UPDATE plants SET last_watered_date = ?, next_watering_date = ? WHERE name = ?",
                (new_last_watered_date, next_watering, plant_name)
            )
            conn.commit()
            print(f"'{plant_name}' updated. New last watered: {new_last_watered_date}, Next watering due: {next_watering}")
        else:
            print(f"Plant '{plant_name}' not found.")
    except ValueError as e:
        print(f"Error with date format: {e}. Please use YYYY-MM-DD.")
    finally:
        conn.close()

Let's add some example plants to your inventory and display them.

In [3]:
# Example: Add some plants
add_plant('Tomato Plant', 'Vegetable', 3, '2023-10-25')
add_plant('Basil', 'Herb', 2, '2023-10-26')
add_plant('Rose', 'Flower', 7, '2023-10-20')
add_plant('Fern', 'Houseplant', 5, '2023-10-23')

# Example: Update a plant's last watered date
update_last_watered('Tomato Plant', '2023-10-27')

Plant 'Tomato Plant' added successfully. Next watering due: 2023-10-28
Plant 'Basil' added successfully. Next watering due: 2023-10-28
Plant 'Rose' added successfully. Next watering due: 2023-10-27
Plant 'Fern' added successfully. Next watering due: 2023-10-28
'Tomato Plant' updated. New last watered: 2023-10-27, Next watering due: 2023-10-30


In [4]:
import pandas as pd

def get_all_plants():
    """Retrieves all plants from the database and returns them as a Pandas DataFrame."""
    conn = sqlite3.connect(DATABASE_NAME)
    df = pd.read_sql_query("SELECT * FROM plants", conn)
    conn.close()
    return df

def display_plants():
    """Displays the current plant inventory."""
    plants_df = get_all_plants()
    if not plants_df.empty:
        print("\n--- Current Plant Inventory ---")
        display(plants_df)
    else:
        print("Your plant inventory is empty.")

display_plants()


--- Current Plant Inventory ---


,id,name,type,watering_frequency_days,last_watered_date,next_watering_date
0,1,Tomato Plant,Vegetable,3,2023-10-27,2023-10-30
1,2,Basil,Herb,2,2023-10-26,2023-10-28
2,3,Rose,Flower,7,2023-10-20,2023-10-27
3,4,Fern,Houseplant,5,2023-10-23,2023-10-28


Next, let's implement the logic to check for plants that are overdue for watering and provide an alert function.

In [5]:
def check_overdue_plants():
    """Checks for plants that are overdue for watering and returns them."""
    conn = sqlite3.connect(DATABASE_NAME)
    cursor = conn.cursor()
    today = datetime.now().date()
    overdue_plants = []

    cursor.execute("SELECT name, next_watering_date FROM plants")
    plants = cursor.fetchall()
    conn.close()

    for name, next_watering_str in plants:
        next_watering = datetime.strptime(next_watering_str, '%Y-%m-%d').date()
        if next_watering <= today:
            overdue_plants.append({'name': name, 'next_watering_due': next_watering_str})

    return pd.DataFrame(overdue_plants) if overdue_plants else pd.DataFrame(columns=['name', 'next_watering_due'])

def alert_overdue_plants():
    """Prints an alert for plants that are overdue for watering."""
    overdue_df = check_overdue_plants()
    if not overdue_df.empty:
        print("\n--- Watering Alerts! ---")
        print("The following plants are overdue for watering:")
        display(overdue_df)
    else:
        print("\nAll plants are watered up to date!")

alert_overdue_plants()


--- Watering Alerts! ---
The following plants are overdue for watering:


,name,next_watering_due
0,Tomato Plant,2023-10-30
1,Basil,2023-10-28
2,Rose,2023-10-27
3,Fern,2023-10-28
